# Quinto Intento: Transformers para Datos Tabulares (Puro PyTorch)

### La Arquitectura Propuesta: Tabular FT-Transformer (Feature Tokenizer Transformer)

Acabas de sugerir el Santo Grial de las redes neuronales tabulares modernas.
En lugar de usar un MLP clásico que mezcla todas las variables a la fuerza, vamos a usar un **Transformer con Self-Attention (Mecanismo de Auto-Atención)**.

**¿Cómo funciona?**
1. **Tokenización:** Toma cada variable del cliente (ej. Edad=25) y en lugar de tratarlo como un simple número, lo proyecta en un "Token" matemático (un vector de 16 dimensiones). Cada columna se convierte en una "palabra" dentro de la "oración" que describe al cliente.
2. **Self-Attention:** El Transformer compara cada variable contra TODAS las demás variables simultáneamente. Así, si descubre que la variable `Saldo` solo importa si la variable `Edad` es mayor a 30, le dará prioridad (Atención) a esa combinación exacta.
3. **Clasificación:** Tras procesar todas las relaciones de atención, la información se condensa y se pasa por un evaluador final que optimiza directamente para Focal Loss (AUC).

In [ ]:
import os
# 🛡️ PARCHE ANTI-DEADLOCK PARA MAC
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings('ignore')
print("✅ Librerías de PyTorch listas.")

In [ ]:
# 1. CARGA DE DATOS PUROS
train_df = pd.read_csv('../../dataset/train.csv')

X = train_df.drop(['id_cliente', 'objetivo'], axis=1)
y = train_df['objetivo']

# Separamos el 10% puro de prueba final
X_train_full, X_test_final, y_train_full, y_test_final = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=42
)

cat_cols = ['ocupacion', 'region', 'canal_adquisicion', 'banda_riesgo', 'dispositivo_principal']
num_cols = [c for c in X.columns if c not in cat_cols]

# Escalado Puro
print("⚙️ Escalando datos para PyTorch...")
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

X_train_rn = preprocessor.fit_transform(X_train_full)
X_test_rn = preprocessor.transform(X_test_final)

# Separamos Validación Limpia Interna (15%)
X_t_feat, X_v_feat, y_t, y_v = train_test_split(
    X_train_rn, y_train_full.values, test_size=0.15, stratify=y_train_full.values, random_state=42
)

# Tensores PyTorch
tensor_X_t = torch.tensor(X_t_feat, dtype=torch.float32)
tensor_y_t = torch.tensor(y_t, dtype=torch.float32).view(-1, 1)
tensor_X_v = torch.tensor(X_v_feat, dtype=torch.float32)
tensor_y_v = torch.tensor(y_v, dtype=torch.float32).view(-1, 1)
tensor_X_test = torch.tensor(X_test_rn, dtype=torch.float32)

print(f"✅ Tensores generados. Dimensiones de entrada: {tensor_X_t.shape[1]} columnas.")

In [ ]:
# 2. LA ARQUITECTURA: TABULAR TRANSFORMER CON SELF-ATTENTION
class TabularTransformer(nn.Module):
    def __init__(self, num_features, d_model=16, nhead=4, num_layers=2):
        super().__init__()
        self.num_features = num_features
        self.d_model = d_model
        
        # Feature Tokenizer: Expande CADA UNA de las 40 variables a un vector de 16 dimensiones
        self.feature_tokenizer = nn.Linear(1, d_model)
        
        # Mecanismo de Atención (El núcleo del Transformer)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=64, dropout=0.2, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Clasificador Final
        self.classifier = nn.Sequential(
            nn.Linear(num_features * d_model, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        # x viene con forma [batch_size, 40 variables]
        # Le agregamos una dimensión extra para tratar cada variable como una unidad aislada [batch, 40, 1]
        x_expanded = x.unsqueeze(2) 
        
        # Pasa por el Tokenizer -> Forma: [batch, 40, d_model=16]
        tokens = self.feature_tokenizer(x_expanded)
        
        # El Transformer aplica Self-Attention. Aquí descubre si Edad se relaciona con Saldo.
        attended_tokens = self.transformer(tokens)
        
        # Aplanamos todas las variables atendidas para pasarlas al clasificador final
        flattened = attended_tokens.reshape(x.size(0), -1)
        
        # Decisión Final
        return self.classifier(flattened)

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss) 
        focal_loss = self.alpha * (1-pt)**self.gamma * bce_loss
        return torch.mean(focal_loss)

In [ ]:
# 3. BUCLE DE ENTRENAMIENTO PURO (AUC EARLY STOPPING)
num_features = tensor_X_t.shape[1]
model = TabularTransformer(num_features=num_features, d_model=16, nhead=4, num_layers=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = FocalLoss(gamma=2.0)

epochs = 200
batch_size = 512
best_auc = 0
patience = 20
patience_counter = 0
best_model_weights = None

print("🚀 Iniciando Entrenamiento de TABULAR TRANSFORMER (Focal Loss + AUC Early Stopping)")
for epoch in range(epochs):
    model.train()
    permutation = torch.randperm(tensor_X_t.size()[0])
    
    for i in range(0, tensor_X_t.size()[0], batch_size):
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = tensor_X_t[indices], tensor_y_t[indices]
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
    model.eval()
    with torch.no_grad():
        val_outputs = model(tensor_X_v)
        val_auc = roc_auc_score(tensor_y_v.numpy(), val_outputs.numpy())
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_model_weights = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            
    if (epoch+1) % 5 == 0:
        print(f"Época {epoch+1:03d} | AUC Validación Limpia: {val_auc:.5f} | Mejor AUC Histórico: {best_auc:.5f}")
        
    if patience_counter >= patience:
        print(f"\n🛑 Early Stopping en la época {epoch+1}. Restaurando el mejor cerebro (AUC={best_auc:.5f})")
        model.load_state_dict(best_model_weights)
        break

In [ ]:
# 4. EVALUACIÓN FINAL TRANSFORMER SOBRE EL 10% TEST PURO
model.eval()
with torch.no_grad():
    final_probs = model(tensor_X_test).numpy()

threshold = np.percentile(final_probs, 85)
final_preds_bin = (final_probs >= threshold).astype(int)

final_auc = roc_auc_score(y_test_final, final_probs)
final_gini = 2.0 * final_auc - 1.0

print("\n=======================================================")
print("📊 RESULTADOS FINALES TRANSFORMER SOBRE SET DE PRUEBA (10%)")
print("=======================================================")
print(f"⭐ AUC FINAL:  {final_auc:.5f}")
print(f"🏆 GINI FINAL: {final_gini:.5f}\n")
print("📉 MATRIZ DE CONFUSIÓN (Threshold Ajustado):")
print(confusion_matrix(y_test_final, final_preds_bin))
print("\n📋 REPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test_final, final_preds_bin))